In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import concurrent.futures
from tqdm import tqdm

# --- LOCAL MODULES ---
import sys
sys.path.append('..')
from src.acquisition import get_best_prefire_layer, download_stitched_home

# --- CONFIGURATION ---
# The fire started Jan 7, 2025. We want the latest map BEFORE this date.
FIRE_DATE = "2025-01-07"

# Paths
INPUT_CSV = "../data/processed/clean_homes.csv"
OUTPUT_DIR = "../data/images"
if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)

print("Configuration loaded.")

In [ ]:
# Get Layer Template
tile_url_template = get_best_prefire_layer(FIRE_DATE)
print(f"Template: {tile_url_template}")

In [ ]:
# Load Data
df = pd.read_csv(INPUT_CSV)
print(f"Queueing up {len(df)} homes...")

# Test Visualization Block
TEST_DIR = "../data/test_zoom_samples"
if not os.path.exists(TEST_DIR): os.makedirs(TEST_DIR)

# Pick 5 random homes
test_df = df.sample(5) 

print(f"Testing Crop 450px (Sharper) on these addresses:")

plt.figure(figsize=(20, 10))

for i, (index, row) in enumerate(test_df.iterrows()):
    try:
        fname_str = row['filename'] if pd.notna(row['filename']) else str(row['id'])
        
        # Run the updated function
        img = download_stitched_home(row['lat'], row['lon'], tile_url_template, zoom=20)
        
        # Save & Show
        img.save(f"{TEST_DIR}/{fname_str}.jpg")
        
        plt.subplot(1, 5, i+1)
        plt.imshow(img)
        plt.title(fname_str[:15], fontsize=9)
        plt.axis('off')
        
    except Exception as e:
        print(f"Error: {e}")

plt.tight_layout()
plt.show()

In [ ]:
# --- MASS DOWNLOAD ---
MAX_WORKERS = 20

def process_one_home(row):
    try:
        # USE ADDRESS FOR FILENAME
        # Fallback to ID if filename is missing
        fname_str = row['filename'] if pd.notna(row['filename']) else str(row['id'])
        save_path = f"{OUTPUT_DIR}/{fname_str}.jpg"
        
        if os.path.exists(save_path):
            return "Skipped"
            
        img = download_stitched_home(row['lat'], row['lon'], tile_url_template, zoom=20)
        
        # Check if we got a valid image (not all black/white)
        extrema = img.convert("L").getextrema()
        if extrema == (0, 0) or extrema == (255, 255):
             return "Blank"

        img.save(save_path)
        return "Success"
            
    except Exception as e:
        return f"Error: {e}"

rows = df.to_dict('records')

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    results = list(tqdm(executor.map(process_one_home, rows), total=len(rows), unit="img"))

print("-" * 30)
print(f"Success: {results.count('Success')}")
print(f"Skipped: {results.count('Skipped')}")
print(f"Errors/Blank: {len(rows) - results.count('Success') - results.count('Skipped')}")